# 13 — Sonic.ma's Voucher Gating Hypothesis

Should the IV-residual mean-reversion strategy on volcanic-rock vouchers be *gated* by a regime filter? This notebook reproduces EDA #13 — testing four candidate gates (rv50, smile RMSE, |imb|, time-of-day) against the canonical R3 voucher MR alpha.

Source script: `scripts/eda_13_sonic_gating.py`. Findings doc: `docs/round_3/research/13_sonic_voucher_gating.md`.

## Setup

Imports, path bootstrapping, and inline plotting. We pull in `BlackScholes` from `src/utils` for IV inversion + vega.

In [ ]:
%matplotlib inline
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils.black_scholes import BlackScholes  # noqa: E402

DATA = ROOT / "data" / "round_3"
DAYS = [0, 1, 2]

## Constants & strike universe

Five live voucher strikes. The smile is fit on the three central strikes (10000/10250/10500); the four "VEV_*" labels are the Discord-relevant test vouchers. Time to expiry decays linearly within the 3-day R3 window from 7 days remaining.

In [ ]:
VFE = "VOLCANIC_ROCK"
STRIKES = {
    "VOLCANIC_ROCK_VOUCHER_9500": 9500,
    "VOLCANIC_ROCK_VOUCHER_9750": 9750,
    "VOLCANIC_ROCK_VOUCHER_10000": 10000,
    "VOLCANIC_ROCK_VOUCHER_10250": 10250,
    "VOLCANIC_ROCK_VOUCHER_10500": 10500,
}
# Discord-targeted strikes ("VEV_4500" = relative offset notation in doc):
# 4500 -> 9500, 5000 -> 10000, 5300 -> ~10250 (closest), 5400 -> 10500
TEST_VOUCHERS = {
    "VEV_4500": "VOLCANIC_ROCK_VOUCHER_9500",
    "VEV_5000": "VOLCANIC_ROCK_VOUCHER_10000",
    "VEV_5300": "VOLCANIC_ROCK_VOUCHER_10250",
    "VEV_5400": "VOLCANIC_ROCK_VOUCHER_10500",
}
SMILE_FIT_STRIKES = [
    "VOLCANIC_ROCK_VOUCHER_10000",
    "VOLCANIC_ROCK_VOUCHER_10250",
    "VOLCANIC_ROCK_VOUCHER_10500",
]
TICKS_PER_DAY = 10_000
DAYS_TO_EXPIRY_AT_R3_START = 7  # day 0 of R3 -> 7 days remaining

## Panel loaders & VFE features

Load semicolon-separated CSVs, pivot to a wide panel keyed by timestamp, and compute two VFE-side features: book-size imbalance and 50-tick annualised realized vol.

In [ ]:
def load_day(day: int) -> pd.DataFrame:
    path = DATA / f"prices_round_3_day_{day}.csv"
    df = pd.read_csv(path, sep=";")
    df["day"] = day
    return df


def mid(df: pd.DataFrame) -> pd.Series:
    return (df["bid_price_1"] + df["ask_price_1"]) / 2.0


def book_imbalance(df: pd.DataFrame) -> pd.Series:
    bid = df[[c for c in df.columns if c.startswith("bid_volume")]].fillna(0).sum(axis=1)
    ask = df[[c for c in df.columns if c.startswith("ask_volume")]].fillna(0).sum(axis=1)
    tot = bid + ask
    return np.where(tot > 0, (bid - ask) / tot, 0.0)


def build_panel(day: int) -> pd.DataFrame:
    """Return wide panel: index=timestamp, cols=mid_<product>, plus VFE features."""
    raw = load_day(day)
    pivots = {}
    for prod in [VFE, *STRIKES.keys()]:
        sub = raw[raw["product"] == prod].set_index("timestamp").sort_index()
        pivots[f"mid_{prod}"] = mid(sub)
    panel = pd.DataFrame(pivots).dropna(how="any")

    vfe_sub = raw[raw["product"] == VFE].set_index("timestamp").sort_index()
    vfe_sub = vfe_sub.loc[panel.index]
    panel["vfe_imb"] = book_imbalance(vfe_sub)

    log_ret = np.log(panel[f"mid_{VFE}"]).diff()
    panel["rv50"] = log_ret.rolling(50).std() * np.sqrt(365 * TICKS_PER_DAY)
    panel["day"] = day
    return panel

## Per-tick smile fit

For each tick, invert IV for the three central strikes, fit a quadratic in moneyness `m = log(K/S)/sqrt(T)`, then evaluate residuals + vega-converted price residuals for every test voucher. RMSE of the central fit becomes our "smile stability" feature.

In [ ]:
def tte_for(day: int, ts: int) -> float:
    """Years to expiry. Voucher expires at end of day 7 (R3 covers days 0..2)."""
    days_left = DAYS_TO_EXPIRY_AT_R3_START - day - (ts / TICKS_PER_DAY)
    return max(days_left / 365.0, 1e-6)


def fit_smile_tick(spot: float, tte: float, prices: dict[str, float]) -> tuple[dict, float]:
    """Fit IV per strike, fit quadratic in moneyness, return (residuals_dict, rmse)."""
    ivs, ms = {}, {}
    for prod in SMILE_FIT_STRIKES:
        K = STRIKES[prod]
        px = prices[prod]
        if px <= max(spot - K, 0) + 1e-6:  # below intrinsic
            continue
        try:
            iv = BlackScholes.implied_vol(px, spot, K, tte)
        except Exception:
            continue
        if not (0.001 < iv < 0.999):
            continue
        ivs[prod] = iv
        ms[prod] = np.log(K / spot) / np.sqrt(tte)
    if len(ivs) < 3:
        return {}, np.nan
    m_arr = np.array([ms[p] for p in ivs])
    iv_arr = np.array([ivs[p] for p in ivs])
    coef = np.polyfit(m_arr, iv_arr, 2)
    fit = np.polyval(coef, m_arr)
    resid = iv_arr - fit
    rmse = float(np.sqrt(np.mean(resid**2)))
    # Now compute residuals for ALL TEST_VOUCHERS using the same fit
    out = {}
    for label, prod in TEST_VOUCHERS.items():
        K = STRIKES[prod]
        px = prices.get(prod)
        if px is None or px <= max(spot - K, 0) + 1e-6:
            continue
        try:
            iv = BlackScholes.implied_vol(px, spot, K, tte)
        except Exception:
            continue
        if not (0.001 < iv < 0.999):
            continue
        m = np.log(K / spot) / np.sqrt(tte)
        iv_fit = float(np.polyval(coef, m))
        vega = BlackScholes.vega(spot, K, tte, iv)
        out[label] = {
            "iv": iv,
            "iv_resid": iv - iv_fit,
            "vega": vega,
            "price_resid": (iv - iv_fit) * vega * 100,  # vega is per 1% vol
            "price": px,
        }
    return out, rmse


def compute_residuals(panel: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ts, row in panel.iterrows():
        spot = row[f"mid_{VFE}"]
        tte = tte_for(int(row["day"]), int(ts))
        prices = {p: row[f"mid_{p}"] for p in STRIKES}
        residuals, rmse = fit_smile_tick(spot, tte, prices)
        rec = {"timestamp": ts, "smile_rmse": rmse}
        for label, info in residuals.items():
            rec[f"{label}_iv_resid"] = info["iv_resid"]
            rec[f"{label}_price_resid"] = info["price_resid"]
            rec[f"{label}_price"] = info["price"]
        rows.append(rec)
    out = pd.DataFrame(rows).set_index("timestamp")
    out["smile_rmse_smoothed"] = out["smile_rmse"].rolling(50, min_periods=10).mean()
    return out

## MR simulator + Sharpe-like

The strategy under test: enter `-sign(price_resid)` 1-lot when `|price_resid| > 1.5`, exit on residual sign-flip, max-hold 50 ticks, force-flat at EOD. `gate_mask` controls *entries only*; existing positions always honour their exit rule.

In [ ]:
def simulate_voucher(prices: pd.Series, price_resid: pd.Series,
                     gate_mask: pd.Series, threshold: float = 1.5,
                     max_hold: int = 50) -> tuple[float, int, list[float]]:
    """1-lot MR strategy on price_resid; entries gated by gate_mask.

    Returns (total_pnl, n_trades, per_trade_pnls).
    """
    pos = 0
    entry_px = 0.0
    entry_sign_resid = 0.0
    hold = 0
    trades: list[float] = []
    idx = prices.index.tolist()
    last_ts = idx[-1]
    for ts in idx:
        px = prices.loc[ts]
        resid = price_resid.loc[ts] if ts in price_resid.index else np.nan
        if pd.isna(px):
            continue
        if pos != 0:
            hold += 1
            flip = (not pd.isna(resid)) and (np.sign(resid) != np.sign(entry_sign_resid))
            timeout = hold >= max_hold
            eod = ts == last_ts
            if flip or timeout or eod:
                pnl = (px - entry_px) * pos
                trades.append(pnl)
                pos = 0
                hold = 0
        if pos == 0 and not pd.isna(resid) and abs(resid) > threshold:
            allowed = bool(gate_mask.loc[ts]) if ts in gate_mask.index else False
            if allowed:
                pos = -int(np.sign(resid))
                entry_px = px
                entry_sign_resid = resid
                hold = 0
    return float(sum(trades)), len(trades), trades


def sharpe_like(trade_pnls: list[float]) -> float:
    if len(trade_pnls) < 2:
        return float("nan")
    arr = np.array(trade_pnls)
    sd = arr.std(ddof=1)
    if sd == 0:
        return float("nan")
    return float(arr.mean() / sd * np.sqrt(len(arr)))

## Build candidate gates

Each gate is a boolean Series over the residual index. Median splits give us low/high pairs for rv50, smile_rmse_smoothed, and |imb|; time-of-day uses a fixed ts=5000 cutoff. `none` is the ungated baseline.

In [ ]:
def build_gates(panel: pd.DataFrame, resid_df: pd.DataFrame) -> dict[str, pd.Series]:
    rv = panel["rv50"].reindex(resid_df.index)
    rmse_s = resid_df["smile_rmse_smoothed"]
    imb = panel["vfe_imb"].reindex(resid_df.index).abs()
    ts_idx = pd.Series(resid_df.index, index=resid_df.index)
    rv_med = rv.median()
    rmse_med = rmse_s.median()
    imb_med = imb.median()
    return {
        "none": pd.Series(True, index=resid_df.index),
        "rv50_low": rv < rv_med,
        "rv50_high": rv >= rv_med,
        "smile_good": rmse_s < rmse_med,
        "smile_bad": rmse_s >= rmse_med,
        "imb_small": imb < max(imb_med, 1e-9),
        "imb_large": imb >= max(imb_med, 1e-9),
        "tod_early": ts_idx < 5000,
        "tod_late": ts_idx >= 5000,
    }

## Run all gates × all vouchers × all days

Aggregate to a gate × voucher PnL pivot, with total trade count and Sharpe-like across all per-trade PnLs in the gate. This reproduces the table in the findings doc.

In [ ]:
def run() -> pd.DataFrame:
    all_results = []
    per_gate_trades: dict[str, list[float]] = {}
    for day in DAYS:
        panel = build_panel(day)
        resid_df = compute_residuals(panel)
        gates = build_gates(panel, resid_df)
        for gate_name, mask in gates.items():
            mask = mask.fillna(False)
            for label, prod in TEST_VOUCHERS.items():
                price_col = f"{label}_price"
                resid_col = f"{label}_price_resid"
                if price_col not in resid_df.columns:
                    pnl, n = 0.0, 0
                    trades: list[float] = []
                else:
                    px_series = resid_df[price_col].dropna()
                    pr_series = resid_df[resid_col].dropna()
                    common = px_series.index.intersection(pr_series.index)
                    pnl, n, trades = simulate_voucher(
                        px_series.loc[common], pr_series.loc[common], mask.loc[common],
                    )
                all_results.append({
                    "day": day, "gate": gate_name, "voucher": label,
                    "pnl": pnl, "n_trades": n,
                })
                per_gate_trades.setdefault(gate_name, []).extend(trades)

    df = pd.DataFrame(all_results)
    pivot = df.pivot_table(index="gate", columns="voucher",
                           values="pnl", aggfunc="sum").fillna(0.0)
    pivot["TOTAL"] = pivot.sum(axis=1)
    pivot["n_trades"] = df.groupby("gate")["n_trades"].sum()
    pivot["sharpe_like"] = pivot.index.map(lambda g: sharpe_like(per_gate_trades.get(g, [])))
    return pivot


table = run()
cols = [c for c in ["VEV_4500", "VEV_5000", "VEV_5300", "VEV_5400",
                    "TOTAL", "n_trades", "sharpe_like"] if c in table.columns]
print(table[cols].round(3).to_string())

## Summary + Key Findings

**Strategy under test**
- Per-tick quadratic smile fit on VEV_5000–5500, IV residual → price via vega, enter `-sign(resid)` 1-lot when `|price_resid| > 1.5`, exit on residual sign-flip, hold ≤ 50 ticks, flat at EOD.
- Universe: VEV_4500, 5000, 5300, 5400. **Only 5300 + 5400 fire** — 4500 = NaN, 5000 = 0 across every gate (deep-ITM / no smile residual).

**Gate results table (3 days, 1-lot)**
- `none` (baseline): VEV_5300 = 35.5, VEV_5400 = 7.0, **TOTAL = 42.5**, n=903, **Sharpe = 0.72**.
- `rv50_low`: 37.0 / -3.0 / 34.0, n=1139, Sharpe 1.14.
- `rv50_high`: 10.5 / 13.0 / 23.5, n=1130, Sharpe 0.94.
- `smile_good`: -4.0 / -2.0 / **-6.0**, n=394, **Sharpe -0.18**.
- `smile_bad`: 37.5 / 10.0 / **+47.5**, n=570, **Sharpe +1.33**.
- `imb_small`: 35.5 / 7.0 / 42.5, n=903, Sharpe 0.72 (identical to baseline).
- `imb_large`: 0 / 0 / 0, n=0 (degenerate).
- `tod_early`: -2.5 / 3.0 / 0.5, n=3, Sharpe 0.28.
- `tod_late`: 36.5 / 2.5 / 39.0, n=902, Sharpe 0.68.

**Three of four gates are dead**
- **|imb| is dead**: VFE book is size-balanced — P50 of `|imb|` = 0.00, P99 = 0.094. Every observation lands in `imb_small`, gate degenerates to "always on."
- **Time-of-day is a confound**: ~all trades fire after ts=5000 because the 100-tick smile-fit warmup eats the early window. "Alpha appears late" is mechanical, not a signal.
- **rv50 is a wash**: median split costs 9–19 of PnL vs ungated; Sharpe rises modestly only because gating halves both numerator and denominator. No regime preference.

**Smile-RMSE gate is the only winner — and it's inverted from the obvious read**
- Gate IN on **instability** (`smile_rmse_smoothed > median`): **PnL 42.5 → 47.5 (+12%), Sharpe 0.72 → 1.33 (+85%)**.
- Gate IN on calm surface: **-6.0 PnL, Sharpe -0.18** — actively negative.
- Mechanism (hypothesis): clean smile = residual is rounding noise (no edge). Noisy smile = a strike has genuinely dislocated and will mean-revert. Consistent with §05's ADF p≈0 stationarity finding.

**Recommendation**
- Adopt **one** gate: `smile_rmse_smoothed > running_median` (50-tick mean, median over a **trailing** 1000-tick window for live).
- Caveat: gate is computed from the same smile being traded → mild look-ahead via the median threshold in this in-sample backtest. **Trailing median is mandatory in live.** Halve the lift estimate for an OOS prior.
- Drop rv50, |imb|, tod — no evidence; imb/tod are degenerate on this data.
- Sonic.ma's gating hypothesis is **partially confirmed**: gating helps, but the useful axis is smile **instability**, not vol regime or book state.

**Plots referenced in the doc**
- `plots/13_gates_overview.png` — rv50, smile RMSE, VFE imbalance time-series + total-PnL bars per gate.
- `plots/13_gate_voucher_heatmap.png` — PnL grid (gate × voucher).